In [11]:
from pathlib import Path
import pandas as pd
from scipy import stats
import numpy as np

cwd = Path.cwd()
root=cwd.parents[2]
print(root)


c:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis


In [12]:
pd.set_option('display.max_columns',None)

In [13]:
ridge_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_ridge_national_residual_programs.csv")
xgb_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_xgb_national_residual_programs.csv")

In [14]:
ridge_residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_error_log,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_year_error_log,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_year_error_log,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score
0,100,3,110422,1,Public,21.0,CA,35.299513,-120.657311,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.023975,"Agriculture, General.",California Polytechnic State University-San Lu...,0.108421,68350.0,61326.997782,7023.002218,84412.0,11.343465,62803.047094,11.047759,21608.952906,0.295706,36,64786.0,11.078845,45934.156654,10.734964,18851.843346,0.343881,18.0,28,high,0.410410,0.381351,0.344075,0.332672,0.114517,0.107777,1.0,1.0,9.0,0.0,8.0,8.0,4.618802,0.171067,0.165734,0.332672
1,100,3,130934,1,Public,24.0,DE,39.187173,-75.540530,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.788999,"Agriculture, General.",Delaware State University,-0.020978,47478.0,48484.495923,-1006.495923,52676.0,10.871915,51187.620284,10.843253,1488.379716,0.028662,24,38873.0,10.568055,37451.301820,10.530797,1421.698180,0.037258,22.0,28,high,0.037961,0.035805,0.029077,0.027552,-0.020759,-0.019706,12.0,10.0,16.0,-2.0,6.0,4.0,3.055050,0.113150,0.165734,0.027552
2,100,3,145813,1,Public,132.0,IL,40.509403,-88.990058,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.934959,"Agriculture, General.",Illinois State University,0.132320,64041.0,56103.791279,7937.208721,63600.0,11.060369,57844.792778,10.965519,5755.207222,0.094850,214,47295.0,10.764160,44089.133345,10.693969,3205.866655,0.070191,205.0,28,high,0.072713,0.072402,0.099494,0.099080,0.141474,0.140478,9.0,7.0,7.0,-2.0,0.0,-2.0,1.154701,0.042767,0.165734,0.099080
3,100,3,149222,1,Public,23.0,IL,37.714193,-89.217273,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.805872,"Agriculture, General.",Southern Illinois University-Carbondale,0.245510,63031.0,49309.494263,13721.505737,57596.0,10.961208,50550.669820,10.830731,7045.330180,0.130477,47,39700.0,10.589106,39173.171714,10.575747,526.828286,0.013359,22.0,28,high,0.013449,0.012685,0.139372,0.135974,0.278273,0.263471,15.0,5.0,1.0,-10.0,-4.0,-14.0,7.211103,0.267078,0.165734,0.135974
4,100,3,149772,1,Public,145.0,IL,40.468086,-90.686899,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.868886,"Agriculture, General.",Western Illinois University,0.102823,58204.0,52516.679939,5687.320061,58333.0,10.973923,53224.130946,10.882267,5108.869054,0.091656,160,48509.0,10.789505,38988.534031,10.571023,9520.465969,0.218482,149.0,28,high,0.244186,0.242667,0.095988,0.095428,0.108295,0.107613,3.0,8.0,10.0,5.0,2.0,7.0,3.605551,0.133539,0.165734,0.107613


In [15]:
# use combined_pct_error which is already your aggregated target
# this is the most relevant test since that's what stage 2 uses



for year in [1,4,5]:

    ridge_errors = np.abs(ridge_residual_df[f'{year}_year_error'])
    xgb_errors   = np.abs(xgb_residual_df[f'{year}_year_error'])
    # align on unit_id
    common = ridge_residual_df.merge(
        xgb_residual_df[['unit_id', f'{year}_year_error']], 
        on='unit_id', 
        suffixes=('_ridge', '_xgb')
    )

    t_stat, p_val = stats.ttest_rel(
        np.abs(common[f'{year}_year_error_ridge']),
        np.abs(common[f'{year}_year_error_xgb'])
    )

    print(f"Ridge mean abs error: {np.abs(common[f'{year}_year_error_ridge']).mean():.4f}")
    print(f"XGB   mean abs error: {np.abs(common[f'{year}_year_error_xgb']).mean():.4f}")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value:     {p_val:.10f}")

Ridge mean abs error: 7478.5828
XGB   mean abs error: 6484.0779
t-statistic: 88.8823
p-value:     0.0000000000
Ridge mean abs error: 8899.0201
XGB   mean abs error: 7462.4940
t-statistic: 109.7135
p-value:     0.0000000000
Ridge mean abs error: 9313.0883
XGB   mean abs error: 7791.7128
t-statistic: 110.4031
p-value:     0.0000000000


In [ ]:
yearly_diff=[]
for year in [1,4,5]:
    actual = ridge_residual_df[f'{year}_year_earning']
    
    ridge_errors = np.abs(actual - ridge_residual_df[f'{year}_year_pred'])
    xgb_errors   = np.abs(actual - xgb_residual_df[f'{year}_year_pred'])

    yearly_diff.append(ridge_errors - xgb_errors)

    t_stat, p_val = stats.ttest_rel(ridge_errors, xgb_errors)
    print(f"Year {year}: Ridge MAE={ridge_errors.mean():.4f} | "
          f"XGB MAE={xgb_errors.mean():.4f} | p={p_val:.4f}")

Year 1: Ridge MAE=7355.7617 | XGB MAE=5987.5118 | p=0.0000
Year 4: Ridge MAE=8774.5110 | XGB MAE=6809.6756 | p=0.0000
Year 5: Ridge MAE=9136.9188 | XGB MAE=7180.6094 | p=0.0000


In [45]:

for year, diff in zip([1, 4, 5], yearly_diff):
    print(f'Year {year}')
    print(f'Mean {diff.mean():.4f} | Std {diff.std():.4f}')


Year 1
Mean 1368.2499 | Std 5746.8267
Year 4
Mean 1964.8354 | Std 7023.4811
Year 5
Mean 1956.3094 | Std 7310.2466


XGBoost achieves a substantially lower MAE than Ridge Regression across all forecast horizons. The average reduction in absolute error is approximately $1.9k; however, the relatively large variance (~$7.3k) indicates that performance gains vary considerably across observations. A paired t-test confirms the difference is highly statistically significant (p << 0.001).

In [48]:

for year, diff in zip([1, 4, 5], yearly_diff):
    print(f'Year {year}')
    print('XGB superior performance pct:',(np.sum(diff > 0) / len(diff)))

Year 1
XGB superior performance pct: 0.5970623153593417
Year 4
XGB superior performance pct: 0.6170518236284823
Year 5
XGB superior performance pct: 0.6201165134321764


XGBoost outperforms Ridge Regression on approximately 62% of observations. While the improvement is not universal, the magnitude of error reduction when XGBoost does outperform is substantial, leading to a significant reduction in average MAE (~$1.9k). This suggests that gains are driven more by large improvements on a subset of cases rather than consistent marginal gains across all predictions.

In [50]:
import plotly.express as px

for year, diff in zip([1, 4, 5], yearly_diff):
    print(f'Year {year}')
    print('skewness:',diff.skew())
    
px.histogram(diff)

Year 1
skewness: 1.477415402307954
Year 4
skewness: 1.6618127710521668
Year 5
skewness: 1.626873050058469


In [49]:
for year, diff in zip([1, 4, 5], yearly_diff):
    print(f'Year {year}')
    mean = diff.mean()
    se = diff.std() / np.sqrt(len(diff))

    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se

    print('Confidence Interval:',ci_low, ci_high)

Year 1
Confidence Interval: 1309.064294182816 1427.435464643152
Year 4
Confidence Interval: 1892.501787525724 2037.1690489416023
Year 5
Confidence Interval: 1881.022378435673 2031.5963372235383


XGBoost consistently outperforms Ridge Regression in MAE, with an average improvement of approximately $1.9k across all horizons. The improvement is statistically significant with tight confidence intervals. While XGBoost only outperforms Ridge on ~62% of observations, the distribution of error differences is positively skewed, indicating that gains are driven by occasional large improvements rather than uniform gains across all predictions.

In [70]:
for year, diff in zip([1, 4, 5], yearly_diff):
    print(f'Year {year}')
    threshold = np.percentile(diff, 90)  # top 10% biggest improvements

    big_wins = ridge_residual_df[diff >= threshold]
    
    print('normal len:',len(ridge_residual_df),'XGB Outpermance',len(big_wins))

    rest = ridge_residual_df[diff < threshold]

    bw_m=big_wins[f'{year}_year_error'].mean()
    r_m=rest[f'{year}_year_error'].mean()

    print(f'big_wins mean: {bw_m}')
    print(f'rest mean: {r_m}')

    print(big_wins[[f'{year}_year_earning']].describe(),'\n')

Year 1
normal len: 36219 XGB Outpermance 3622
big_wins mean: 3962.6002711071965
rest mean: 705.8380421657431
       1_year_earning
count     3622.000000
mean     68981.893153
std      32957.727184
min       4752.000000
25%      48253.250000
50%      66985.000000
75%      85080.500000
max     250706.000000 

Year 4
normal len: 36219 XGB Outpermance 3622
big_wins mean: 5870.467890880301
rest mean: 836.5492015650365
       4_year_earning
count     3622.000000
mean     90694.988956
std      39054.874277
min      12391.000000
25%      65500.250000
50%      84172.000000
75%     109861.250000
max     336392.000000 

Year 5
normal len: 36219 XGB Outpermance 3622
big_wins mean: 6245.825549428804
rest mean: 757.3256204534312
       5_year_earning
count     3622.000000
mean     90684.501933
std      40818.969515
min      14226.000000
25%      64808.250000
50%      83563.500000
75%     110347.250000
max     384547.000000 

